In [1]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

In [2]:
in_data = "./DATASET"
out_data = "./csvs/"
os.makedirs(out_data, exist_ok=True)

## Current Threshold: redefining/restricting operational zone
c_thresh = 4300

## Thresholding distance to preceeding point to detemrine continuous scan
delta_thresh = 0.035

## Looping through each layer and creating preprocessed csvs
for f in os.listdir(in_data):
    in_path = os.path.join(in_data, f)
    if os.path.exists(in_path) and os.path.basename(in_path).endswith('.hdf5'):
        tag = f.split(".")[0]
        out_path = os.path.join(out_data, tag + ".csv")
        
        file_info = h5py.File(in_path, 'r')
        file_data = file_info['OpenData']
        x = file_data[0]
        y = file_data[1]
        power = file_data[2]
        speed = file_data[3]
        dia = file_data[4]
        cur = file_data[5]
        sig = file_data[6]
        id1 = file_data[7]
        id2 = file_data[8]
        c1 = file_data[9]
        c2 = file_data[10]

        ## N x 11 datapoints per layer
        data = np.vstack([x, y, power, speed, dia, cur, sig, id1, id2, c1, c2]).T

        ## Filter out where the current is not in operational zone
        opzone_condition = data[:, 5] > c_thresh
        data = data[opzone_condition]
        
        ## Determine continuous scans and grouping with scan numbers
        x = data[:, 0]
        y = data[:, 1]
        deltas = np.zeros_like(x)
        scan_nums = np.zeros_like(x, dtype=int)
        scan = int(0)
        
        
        for i in range(1, len(deltas)):
            deltas[i] = np.sqrt((x[i]-x[i-1])**2 + (y[i]-y[i-1])**2)
            if deltas[i] > delta_thresh:
                scan+= int(1)
            scan_nums[i] = scan 
            

        ## Expanding data array to include euclidean distance from previous point and scan number
        deltas = np.expand_dims(deltas, -1)
        scan_nums = np.expand_dims(scan_nums, -1)
        data = np.hstack([data, deltas, scan_nums])

        ## Save to CSV
        headers = [
            "x", "y", "power", "speed", "spot_diameter", "laser_current", 
            "signal", "ID1", "ID2", "C1", "C2", "delta", "scan_number"]
        df = pd.DataFrame(data, columns=headers)
        df.to_csv(out_path)


In [4]:
## Testing - Sorting Directory Contents + Visualizing scan numbers
data_dir = "./csvs/"
scan_imgs_dir = "./scan_imgs/"
os.makedirs(scan_imgs_dir, exist_ok=True)

files = os.listdir(data_dir)
print("len(files)", len(files))
files = [f for f in files if f.endswith(".csv")]
print("len(files)", len(files))
files.sort(key=lambda x: int(x.split(".")[0][5:]))

## Visualizing scan numbers
hues = ["red", "blue", "green", "purple", "orange", "black", "grey", "brown"]
for f_idx in range(len(files)): 
    file_path = os.path.join(data_dir, files[f_idx])
    tag = files[f_idx].split(".")[0]
    img_path = os.path.join(scan_imgs_dir, tag+".png")
    data_df = pd.read_csv(file_path)

    x = data_df["x"].to_numpy()
    y = data_df["y"].to_numpy()
    scan_nums = data_df["scan_number"].to_numpy()
    coloring = []
    for i in range(len(x)):
        col_idx = int(scan_nums[i]) % len(hues)
        coloring.append(hues[col_idx])

    j=len(x)
    # print(j)
    plt.figure()
    plt.scatter(x[:j], y[:j], c=coloring[:j], s =2)
    plt.savefig(img_path, dpi=300)
    plt.close()

len(files) 380
len(files) 379
